# <font color="#418FDE" size="6.5" uppercase>**Spectral Hierarchical**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply Spectral Clustering using appropriate affinity construction. 
- Fit Agglomerative Clustering with different linkage and stopping settings. 
- Use connectivity constraints and Feature Agglomeration for structured clustering. 


## **1. Spectral Clustering**

### **1.1. Spectral Clustering Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_01.jpg?v=1784021333" width="250">



>* Treats data as connected relationship networks
>* Finds non-round clusters through affinity patterns

>* Transform connectivity into easier clustering space
>* Group points by shared graph relationships

>* Captures complex shapes through similarity graphs
>* Requires careful affinity choices for meaningful clusters



In [ ]:
#@title Python Code - Spectral Clustering Basics

# This example compares two clustering views.
# Spectral clustering uses a nearest-neighbor affinity graph.
# The plot shows curved clusters separated correctly.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

# Make a small curved dataset with known labels.
features, true_labels = make_moons(
    n_samples=300,
    noise=0.06,
    random_state=42,
)

# Validate the generated data before clustering.
if features.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 features.")

# Fit spectral clustering using local neighbor connections.
model = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=10,
    random_state=42,
)

predicted_labels = model.fit_predict(features)
score = adjusted_rand_score(true_labels, predicted_labels)

print(f"scikit-learn version: {sklearn_version}")
print("Affinity used: nearest_neighbors with 10 neighbors")
print(f"Adjusted Rand score versus known moons: {score:.3f}")

# Plot the clusters found from the affinity graph.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=predicted_labels,
    cmap="viridis",
    s=35,
)

ax.set_title("Spectral clustering on two curved moons")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



### **1.2. Affinity Matrix Construction**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_02.jpg?v=1784021334" width="250">



>* Affinity graphs define point connections.
>* They reveal non-spherical cluster structure.

>* Convert distances into scaled similarity connections
>* Use nearest neighbors to preserve local structure

>* Normalize features and choose suitable similarity measures
>* Build graphs using meaningful domain relationships



In [ ]:
#@title Python Code - Affinity Matrix Construction

# This example builds two affinity matrices.
# Affinity choices change spectral clustering behavior.
# The plot compares local and global connections.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import StandardScaler

# Curved moon shapes are useful for graph-based clustering.
points, true_groups = make_moons(n_samples=220, noise=0.06, random_state=42)
scaled_points = StandardScaler().fit_transform(points)

# Validate the small two-feature dataset before clustering.
if scaled_points.shape != (220, 2):
    raise ValueError("Expected 220 rows and 2 features.")

# A dense RBF affinity connects every pair with fading strength.
distances = pairwise_distances(scaled_points)
gamma = 1.0
rbf_affinity = np.exp(-gamma * distances ** 2)

# A nearest-neighbor affinity keeps only local graph connections.
neighbors = 10
neighbor_graph = kneighbors_graph(scaled_points, neighbors, mode="connectivity")
nearest_affinity = 0.5 * (neighbor_graph + neighbor_graph.T)

# Spectral clustering can use either precomputed affinity matrix.
model = SpectralClustering(
    n_clusters=2,
    affinity="precomputed",
    assign_labels="kmeans",
    random_state=42,
)

# Fit once with the local nearest-neighbor affinity.
cluster_labels = model.fit_predict(nearest_affinity)

# Summaries show how sparse and dense affinities differ.
rbf_density = np.mean(rbf_affinity > 0.05)
nearest_density = nearest_affinity.nnz / nearest_affinity.shape[0] ** 2
print(f"scikit-learn version: {sklearn_version}")
print(f"RBF affinity density above 0.05: {rbf_density:.3f}")
print(f"Nearest-neighbor affinity density: {nearest_density:.3f}")
print(f"Cluster sizes from nearest-neighbor affinity: {np.bincount(cluster_labels)}")

# The scatter plot shows clusters found from local connections.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    scaled_points[:, 0],
    scaled_points[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=35,
)

# Labels describe the standardized feature space.
ax.set_title("Spectral Clustering with Nearest-Neighbor Affinity")
ax.set_xlabel("Standardized feature 1")
ax.set_ylabel("Standardized feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



### **1.3. Eigen Solver Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_03.jpg?v=1784021336" width="250">



>* Eigenvectors reveal easier cluster-separating embeddings
>* Choose direct or iterative solvers by scale

>* Match solvers to matrix size and sparsity
>* Approximate methods need careful convergence checks

>* Match solvers to data size and graph
>* Test stability across reasonable solver settings



In [ ]:
#@title Python Code - Eigen Solver Choices

# Compare spectral clustering eigen solver choices.
# Solver choice affects speed and labels.
# A moons dataset shows practical similarity.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score

# Create a small nonlinear clustering dataset.
features, true_labels = make_moons(
    n_samples=300,
    noise=0.06,
    random_state=42,
)

# Validate the expected two-column feature matrix.
if features.shape != (300, 2):
    raise ValueError("Expected 300 samples with two features.")

# Fit spectral clustering with a dense eigen solver.
dense_model = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=10,
    eigen_solver="arpack",
    random_state=42,
)

# Fit spectral clustering with a sparse iterative solver.
sparse_model = SpectralClustering(
    n_clusters=2,
    affinity="nearest_neighbors",
    n_neighbors=10,
    eigen_solver="lobpcg",
    random_state=42,
)

# Compare labels from both solver choices.
dense_labels = dense_model.fit_predict(features)
sparse_labels = sparse_model.fit_predict(features)
solver_agreement = adjusted_rand_score(dense_labels, sparse_labels)
true_score = adjusted_rand_score(true_labels, dense_labels)

print("scikit-learn version:", sklearn.__version__)
print("Agreement between eigen solvers:", round(solver_agreement, 3))
print("Dense solver score versus true moons:", round(true_score, 3))

# Plot one solver result to connect labels with geometry.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(
    features[:, 0],
    features[:, 1],
    c=dense_labels,
    cmap="viridis",
    s=28,
)

ax.set_title("Spectral clustering with ARPACK eigen solver")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.show()



## **2. Agglomerative Methods**

### **2.1. Agglomerative Clustering**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_01.jpg?v=1784021331" width="250">



>* Starts with individual points, then merges clusters
>* Reveals hierarchical structure at multiple levels

>* Choose fixed clusters or distance-based stopping
>* Match stopping rules to analysis goals

>* Merge history reveals cluster structure clearly
>* Prepare data carefully to avoid misleading hierarchies



In [ ]:
#@title Python Code - Agglomerative Clustering

# This example fits agglomerative clustering models.
# It compares fixed clusters with distance stopping.
# The plot shows how stopping changes labels.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

# Generate a small dataset with visible cluster structure.
features, true_groups = make_blobs(
    n_samples=90,
    centers=3,
    cluster_std=0.75,
    random_state=42,
)

# Scale features so distances treat both axes fairly.
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Validate the data shape before fitting the model.
if scaled_features.shape != (90, 2):
    raise ValueError("Expected 90 rows and 2 features.")

# Fit one model that stops at exactly three clusters.
fixed_model = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward",
)
fixed_labels = fixed_model.fit_predict(scaled_features)

# Fit another model that stops using a distance threshold.
threshold_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.6,
    linkage="ward",
)
threshold_labels = threshold_model.fit_predict(scaled_features)

fixed_count = len(np.unique(fixed_labels))
threshold_count = len(np.unique(threshold_labels))

print("scikit-learn version:", sklearn.__version__)
print("Fixed n_clusters result:", fixed_count, "clusters")
print("Distance threshold result:", threshold_count, "clusters")

# Plot the threshold-based result for visual interpretation.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    scaled_features[:, 0],
    scaled_features[:, 1],
    c=threshold_labels,
    cmap="viridis",
    s=55,
)

ax.set_title("Agglomerative Clustering with Distance Stopping")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



### **2.2. Linkage Strategies**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_02.jpg?v=1784021327" width="250">



>* Linkage decides which clusters merge next
>* Different rules create different cluster shapes

>* Single and complete use nearest or farthest pairs
>* Average and Ward create balanced compact clusters

>* Match linkage to data and goals
>* Cut dendrograms using meaningful stopping rules



In [ ]:
#@title Python Code - Linkage Strategies

# Compare linkage choices in agglomerative clustering.
# Linkage changes which nearby groups merge first.
# The plot shows different cluster assignments.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs

# Create compact groups plus bridge points for linkage comparison.
blob_points, _ = make_blobs(
    n_samples=54, centers=[[-3, 0], [3, 0]], cluster_std=0.55, random_state=42
)

bridge_points = np.array([[-1.2, 0.05], [0.0, -0.05], [1.2, 0.05]])
points = np.vstack([blob_points, bridge_points])

# Validate the small two-feature dataset before clustering.
if points.shape != (57, 2):
    raise ValueError("Expected 57 rows and 2 numeric features.")

# Fit the same estimator type with different linkage rules.
linkage_names = ["single", "complete", "average", "ward"]
cluster_labels = []

for linkage_name in linkage_names:
    model = AgglomerativeClustering(n_clusters=2, linkage=linkage_name)
    labels = model.fit_predict(points)
    cluster_labels.append(labels)

# Print a compact summary of how each linkage split the data.
print(f"scikit-learn version: {sklearn_version}")
print("Linkage strategy -> cluster sizes")

for linkage_name, labels in zip(linkage_names, cluster_labels):
    sizes = np.bincount(labels)
    print(f"{linkage_name:8s} -> {sizes[0]} and {sizes[1]} points")

# Plot one linkage result to connect the summary with geometry.
colors = np.array(["tab:blue", "tab:orange"])
fig, ax = plt.subplots(figsize=(7, 5))

chosen_index = linkage_names.index("single")
ax.scatter(points[:, 0], points[:, 1], c=colors[cluster_labels[chosen_index]], s=55)
ax.set_title("Single linkage can follow bridge points")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
plt.show()



### **2.3. Connectivity Constraints**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_03.jpg?v=1784021329" width="250">



>* Limit merges to connected neighboring observations
>* Preserve coherent clusters in structured data

>* Blend similarity with real-world structure
>* Prevent contextually invalid cluster merges

>* Fewer allowed merges can speed clustering
>* Choose constraints matching real data structure



In [ ]:
#@title Python Code - Connectivity Constraints

# This example shows constrained agglomerative clustering.
# Connectivity allows only nearby points to merge.
# The plot compares structured and unstructured clusters.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph

# A fixed generator makes the synthetic example repeatable.
rng = np.random.default_rng(42)

# Two noisy arcs create a shape where structure matters.
angles = np.linspace(0.0, np.pi, 80)
upper_arc = np.column_stack((np.cos(angles), np.sin(angles)))
lower_arc = np.column_stack((np.cos(angles), -np.sin(angles) - 0.35))

# Small noise keeps the arcs realistic but still clear.
points = np.vstack((upper_arc, lower_arc))
points = points + rng.normal(0.0, 0.04, size=points.shape)

# Validate the simple two-dimensional clustering input.
if points.shape != (160, 2):
    raise ValueError("Expected 160 points with two features.")

# The graph says each point may merge through nearby neighbors.
connectivity = kneighbors_graph(points, n_neighbors=8, include_self=False)

# Agglomerative clustering now respects the neighbor graph.
model = AgglomerativeClustering(
    n_clusters=2,
    linkage="ward",
    connectivity=connectivity,
)

# Fit the model and obtain one cluster label per point.
labels = model.fit_predict(points)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Connectivity graph edges: {connectivity.nnz}")
print(f"Cluster sizes: {np.bincount(labels).tolist()}")

# The colors show clusters formed under the connectivity constraint.
fig, ax = plt.subplots(figsize=(6, 4))
scatter = ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="viridis", s=35)

# Labels make the structured clustering result easier to read.
ax.set_title("Agglomerative clustering with connectivity constraints")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")

plt.show()



## **3. Structured Hierarchy Practice**

### **3.1. Stopping Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_01.jpg?v=1784021338" width="250">



>* Stopping rules end hierarchical merging meaningfully
>* Choose granularity that supports useful decisions

>* Choose fixed cluster counts for practical needs
>* Use distance thresholds to preserve similarity

>* Constraints preserve known structure during merging
>* Choose stopping levels balancing clarity and fidelity



In [ ]:
#@title Python Code - Stopping Criteria

# This example compares hierarchy stopping criteria.
# Connectivity keeps neighboring points merged together.
# Feature agglomeration shows structured feature compression.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import FeatureAgglomeration
from sklearn.neighbors import kneighbors_graph

# A small line-shaped dataset makes adjacency easy to see.
positions = np.arange(12).reshape(-1, 1)
values = np.array([0.0, 0.2, 0.1, 3.0, 3.2, 3.1, 6.0, 6.1, 6.3, 9.0, 9.1, 9.2])
X = np.column_stack([positions.ravel(), values])

# Validate the simple shape before clustering.
if X.shape != (12, 2):
    raise ValueError("Expected 12 observations with 2 features.")

# Connectivity allows only neighboring observations to merge.
connectivity = kneighbors_graph(positions, n_neighbors=2, include_self=False)

# Fixed cluster count stops after exactly three groups remain.
fixed_model = AgglomerativeClustering(
    n_clusters=3,
    linkage="ward",
    connectivity=connectivity,
)
fixed_labels = fixed_model.fit_predict(X)

# Distance threshold stops when the next merge is too dissimilar.
threshold_model = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=2.0,
    linkage="ward",
    connectivity=connectivity,
)
threshold_labels = threshold_model.fit_predict(X)

# Feature agglomeration compresses related columns into two groups.
feature_data = np.column_stack([
    values,
    values + 0.05,
    10 - values,
    10.1 - values,
])
feature_model = FeatureAgglomeration(n_clusters=2)
compressed = feature_model.fit_transform(feature_data)

print("scikit-learn version:", sklearn.__version__)
print("Fixed n_clusters=3 labels:", fixed_labels.tolist())
print("Distance threshold=2.0 labels:", threshold_labels.tolist())
print("Feature groups:", feature_model.labels_.tolist())
print("Compressed feature shape:", compressed.shape)

# The plot shows how the fixed stopping rule partitions the line.
fig, ax = plt.subplots(figsize=(7, 4))
scatter = ax.scatter(positions.ravel(), values, c=fixed_labels, s=80)
ax.set_title("Agglomerative stopping with connectivity")
ax.set_xlabel("Neighbor position")
ax.set_ylabel("Measured value")
ax.legend(*scatter.legend_elements(), title="Cluster")
plt.show()



### **3.2. Feature Agglomeration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_02.jpg?v=1784021340" width="250">



>* Groups similar features instead of observations
>* Reduces dimensions while preserving useful structure

>* Connectivity limits merges to related features
>* Clusters reflect domain structure and data patterns

>* Reduce features into representative groups
>* Validate clusters with domain knowledge



In [ ]:
#@title Python Code - Feature Agglomeration

# This example clusters related columns together.
# Connectivity keeps neighboring features merging first.
# The output shows reduced structured feature groups.

import numpy as np
import sklearn
from sklearn.cluster import FeatureAgglomeration
from sklearn.feature_extraction.image import grid_to_graph

# Create samples with six ordered sensor features.
rng = np.random.default_rng(42)
base_left = rng.normal(size=(40, 1))
base_right = rng.normal(size=(40, 1))

# Neighboring columns share similar hidden signals.
left_features = base_left + rng.normal(scale=0.15, size=(40, 3))
right_features = base_right + rng.normal(scale=0.15, size=(40, 3))
X = np.hstack([left_features, right_features])

# Validate the small teaching matrix shape.
if X.shape != (40, 6):
    raise ValueError("Expected 40 samples and 6 features.")

# Connect only adjacent features along one sensor line.
connectivity = grid_to_graph(n_x=1, n_y=6)

# Agglomerate six original features into two structured groups.
agglomerator = FeatureAgglomeration(
    n_clusters=2,
    connectivity=connectivity,
    linkage="ward"
)
X_reduced = agglomerator.fit_transform(X)

# Summarize which original columns joined each feature cluster.
labels = agglomerator.labels_
groups = []
for cluster_id in sorted(set(labels)):
    members = np.where(labels == cluster_id)[0].tolist()
    groups.append(members)

print("scikit-learn version:", sklearn.__version__)
print("Original feature count:", X.shape[1])
print("Reduced feature count:", X_reduced.shape[1])
print("Feature groups:", groups)
print("First reduced row:", np.round(X_reduced[0], 2).tolist())



### **3.3. Interpreting Cluster Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_03.jpg?v=1784021342" width="250">



>* Hierarchy shows constrained merging paths
>* Constraints reveal local patterns and boundaries

>* Early merges show strong local similarity
>* Later merges reveal broader constrained structures

>* Group related features to simplify data
>* Interpret clusters using statistics and domain meaning



In [ ]:
#@title Python Code - Interpreting Cluster Structure

# This example interprets constrained hierarchical clustering.
# Connectivity limits which neighboring points may merge.
# The plot reveals structured cluster boundaries.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import AgglomerativeClustering
from sklearn.neighbors import kneighbors_graph

# A small grid represents locations with measured values.
rng = np.random.default_rng(42)
side_length = 8
row_values, col_values = np.indices((side_length, side_length))

# Nearby cells have similar values, with one vertical change.
measurements = col_values.astype(float)
measurements[:, 4:] = measurements[:, 4:] + 3.0
measurements = measurements + rng.normal(0.0, 0.25, measurements.shape)

# Each observation stores row, column, and measured value.
locations = np.column_stack((row_values.ravel(), col_values.ravel()))
values = measurements.ravel().reshape(-1, 1)

# The connectivity graph allows only local neighbor merges.
connectivity = kneighbors_graph(locations, n_neighbors=4, include_self=False)
connectivity = 0.5 * (connectivity + connectivity.T)

# Agglomerative clustering now respects the grid structure.
model = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward",
    connectivity=connectivity,
)
cluster_labels = model.fit_predict(values)

# Validate that every grid cell received one cluster label.
if cluster_labels.shape[0] != side_length * side_length:
    raise ValueError("Each grid cell should receive one cluster label.")

# Reshape labels so the hierarchy can be interpreted spatially.
cluster_grid = cluster_labels.reshape(side_length, side_length)
unique_clusters = np.unique(cluster_labels).size

print("scikit-learn version:", sklearn.__version__)
print("Grid cells clustered:", cluster_labels.size)
print("Structured clusters found:", unique_clusters)
print("Connectivity idea: only nearby grid cells were eligible to merge.")

# The image shows how constrained merges form contiguous regions.
fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(cluster_grid, cmap="tab10", interpolation="nearest")

ax.set_title("Interpreting a Connectivity-Constrained Hierarchy")
ax.set_xlabel("Grid column")
ax.set_ylabel("Grid row")

plt.colorbar(image, ax=ax, label="Cluster label")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Spectral Hierarchical**</font>


In this lecture, you learned to:
- Apply Spectral Clustering using appropriate affinity construction. 
- Fit Agglomerative Clustering with different linkage and stopping settings. 
- Use connectivity constraints and Feature Agglomeration for structured clustering. 

In the next Module (Module 23), we will go over 'Density Biclustering'